# ⚡ EnergyPlus Simulation: VS Code + Google Colab Extension

This notebook is optimized for execution via **VS Code with the Google Colab extension**.

## 🚀 Hybrid Workflow
1. **Frontend**: VS Code (Local) - Code editing, IntelliSense, Copilot.
2. **Backend**: Google Colab Runtime (Remote) - Linux execution, EnergyPlus installation, GCS access.

## 📋 Prerequisites
- [Google Colab for VS Code](https://marketplace.visualstudio.com/items?itemName=google.colab) extension installed.
- Active connection to a Colab Runtime (Connect to Colab).
- Access to the `eplus-colab-cloud-data` GCS bucket configured.

## 🎯 Architecture
This notebook works directly with files stored in Google Cloud Storage:
- **IDF Models** and **EPW weather files** are downloaded from the bucket
- **Simulations** are executed locally on the Colab runtime
- **Results** are automatically uploaded back to the bucket

No need to clone repositories or configure GitHub tokens.

---

## 🆕 Update: New GCS Bucket Structure

**Date**: February 2026

This notebook has been updated to work with the new organized bucket structure.

### 🎯 Main Changes:
- ✅ **Organized inputs**: IDF models in `models/`, EPW files in `weather/`
- ✅ **Centralized outputs**: All results in `results/` with timestamps
- ✅ **Updated Stage-In**: Automatic download from correct folders
- ✅ **Enhanced Stage-Out**: Upload only outputs (ignores inputs)
- ✅ **New utilities**: Functions to explore and manage bucket files
- ✅ **Simplified**: Removed dependency on repository cloning

### 📝 How to Use:
1. Run the cells in order (1 → 10)
2. Files will be downloaded automatically from `models/` and `weather/`
3. Results uploaded to `results/colab_vs_code_simulation_{timestamp}/`
4. Use cells 9 and 10 to explore the available files in the bucket

---

In [ ]:
# @title 1. Authentication and GCP Project Configuration
import os
import sys
import warnings
import re
import time
from typing import Optional

# --- Configuration ---
PROJECT_ID = 'eplus-colab-cloud'  # @param {type:"string"}

def authenticate_colab_session(project_id: str) -> None:
    """Authenticates the Colab session and configures the GCP project."""
    print(f"🔐 Configuring project: {project_id}")
    os.environ['GOOGLE_CLOUD_PROJECT'] = project_id

    try:
        import subprocess

        print("🚀 Starting authentication...")
        print("⚠️ INSTRUCTIONS:")
        print("1. CLICK on the long URL that will appear BELOW (Cell Output).")
        print("2. Login in the browser and copy the code.")
        print("3. Paste the code into the input box at the TOP of VS Code and press Enter.")

        # Executes gcloud with manual I/O control to ensure the URL is clickable
        # and does not get stuck inside the VS Code input modal
        # FIX: Removes old credentials to avoid the 'Do you want to continue (Y/n)?' prompt
        if 'GOOGLE_APPLICATION_CREDENTIALS' in os.environ:
            print("🧹 Clearing old credentials to force new authentication...")
            del os.environ['GOOGLE_APPLICATION_CREDENTIALS']

        # Configures environment to prevent line breaking in the URL (truncation)
        env = os.environ.copy()
        env['COLUMNS'] = '2000' # Extra width to prevent line breaks in the URL

        # Returns to the default command. The 'Missing scope' error was caused by the URL truncation.
        # With COLUMNS=2000, the default URL (with correct scopes) will be fully generated.
        cmd = [
            'gcloud', 'auth', 'application-default', 'login', '--no-launch-browser'
        ]

        with subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            stdin=subprocess.PIPE,
            text=True,
            bufsize=1,
            universal_newlines=True,
            env=env  # Passes the environment with COLUMNS=2000
        ) as p:
            buffer = ""
            while True:
                char = p.stdout.read(1)
                if not char:
                    break

                sys.stdout.write(char)
                sys.stdout.flush() # Ensures the user sees the link
                buffer += char

                # Detects the code prompt
                if "verification code" in buffer.lower() or "authorization code" in buffer.lower():
                    print("\n\n⚠️  The input box will appear at the TOP of VS Code now! ⚠️")
                    print("⏳ Waiting 2 seconds for you to click the link...")
                    time.sleep(2)
                    user_code = input("Paste the code in the top box: ")
                    p.stdin.write(user_code.strip() + "\n")
                    p.stdin.flush()
                    buffer = "" # Clears buffer after processing input

            p.wait()
            if p.returncode != 0:
                raise subprocess.CalledProcessError(p.returncode, cmd)

        print("✅ User authenticated successfully (ADC).")
    except ImportError:
        print("⚠️ Local runtime detected. Using system credentials (ADC).")
    except Exception as e:
        print(f"❌ Authentication error: {e}")
        print("   If you canceled the prompt, try running the cell again.")

    # Attempts to configure via gcloud for CLI compatibility
    try:
        import subprocess
        subprocess.run(['gcloud', 'config', 'set', 'project', project_id], check=True, capture_output=True)
        print(f"✅ GCP Project configured: {project_id}")
    except Exception as e:
        warnings.warn(f"Could not configure gcloud CLI (non-critical): {e}")

authenticate_colab_session(PROJECT_ID)


In [ ]:
# @title 2. Resource Diagnostics (CPU/RAM)
import psutil
import os

print("--- Execution Environment Diagnostics ---")
try:
    # CPU
    cpu_count = os.cpu_count()
    print(f"🧠 Logical CPUs: {cpu_count}")

    # RAM
    ram_gb = psutil.virtual_memory().total / 1e9
    print(f"💾 Total RAM Memory: {ram_gb:.2f} GB")

    if ram_gb < 20:
        print("⚠️  Warning: Default runtime (Standard RAM). For large models, consider High-RAM.")
    else:
        print("✅  High Memory Runtime (High-RAM) detected.")

    # GPU (Optional for EnergyPlus, but good to know)
    gpu_info = get_ipython().getoutput('nvidia-smi')
    gpu_info = '\n'.join(gpu_info)
    if 'failed' in gpu_info or 'not found' in gpu_info:
        print("ℹ️  No dedicated GPU detected (OK for EnergyPlus).")
    else:
        print("🚀 GPU Detected (Available for ML/TensorFlow).")
except Exception as e:
    print(f"Diagnostic error: {e}")


In [ ]:
# @title 3. GCS Bucket and Files Definition
from google.cloud import storage
from typing import List

BUCKET_NAME = 'eplus-colab-cloud-data' # @param {type:"string"}

# Bucket Structure:
# gs://eplus-colab-cloud-data/
#   ├── models/          ← IDF Files
#   ├── weather/         ← EPW Files
#   ├── results/         ← Simulation results
#   ├── scripts/         ← Installation scripts
#   └── notebooks/       ← Notebooks

# Input files (with paths relative to the bucket)
IDF_FILE = 'models/5ZoneAirCooled.idf'
EPW_FILE = 'weather/USA_IL_Chicago-OHare.Intl.AP.725300_TMY3.epw'

print(f"🎯 Target Bucket: gs://{BUCKET_NAME}")
print(f"📁 IDF File: {IDF_FILE}")
print(f"🌦️  EPW File: {EPW_FILE}")

# Validation and bucket inspection using Cloud Storage API
try:
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(BUCKET_NAME)

    # Checks if the bucket exists and is accessible
    if bucket.exists():
        print("✅ Bucket access confirmed.")

        # Lists available files in main folders
        print("\n📋 Bucket structure:")

        folders = ['models/', 'weather/']
        for folder in folders:
            print(f"\n📂 {folder}")
            blobs = list(client.list_blobs(BUCKET_NAME, prefix=folder, delimiter='/'))

            # Filters only files (no subfolders)
            files = [blob.name for blob in blobs if not blob.name.endswith('/') and blob.name != folder]

            if files:
                for file_path in files[:5]:  # Shows up to 5 files
                    filename = file_path.split('/')[-1]
                    blob = bucket.blob(file_path)
                    # Fetches metadata (size)
                    blob.reload()
                    size_mb = blob.size / (1024 * 1024) if blob.size else 0
                    print(f"  • {filename} ({size_mb:.2f} MB)")

                if len(files) > 5:
                    print(f"  ... and {len(files) - 5} more file(s)")
            else:
                print("  (empty)")

        # Checks if specified files exist
        print("\n🔍 Checking input files:")
        idf_blob = bucket.blob(IDF_FILE)
        epw_blob = bucket.blob(EPW_FILE)

        if idf_blob.exists():
            idf_blob.reload()
            print(f"  ✅ {IDF_FILE} ({idf_blob.size / (1024 * 1024):.2f} MB)")
        else:
            print(f"  ⚠️ {IDF_FILE} not found!")

        if epw_blob.exists():
            epw_blob.reload()
            print(f"  ✅ {EPW_FILE} ({epw_blob.size / (1024 * 1024):.2f} MB)")
        else:
            print(f"  ⚠️ {EPW_FILE} not found!")

    else:
        print(f"❌ Bucket '{BUCKET_NAME}' does not exist or is not accessible.")

except Exception as e:
    print(f"❌ Error accessing bucket: {e}")
    print("   Ensure authentication (cell 1) was executed correctly.")


## 🛠️ EnergyPlus v25.1.0 Installation
Installs the simulation engine on the remote Linux VM.

In [ ]:
# @title 4. Install EnergyPlus
from pathlib import Path
import subprocess

EPLUS_VERSION = "25.1.0"
EPLUS_URL = 'https://github.com/NREL/EnergyPlus/releases/download/v25.1.0/EnergyPlus-25.1.0-68a4a7c774-Linux-Ubuntu22.04-x86_64.run'
INSTALL_PATH = Path('/eplus')

def install_energyplus(url: str, install_path: Path) -> None:
    if install_path.exists():
        print(f"✅ EnergyPlus already installed in {install_path}")
        return

    print(f"⬇️ Downloading EnergyPlus v{EPLUS_VERSION}...")
    installer = Path('/tmp/ep_installer.run')
    subprocess.run(['wget', '-q', '-O', str(installer), url], check=True)
    subprocess.run(['chmod', '+x', str(installer)], check=True)

    print("📦 Installing system dependencies...")
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    deps = ['libxcb-icccm4', 'libxcb-image0', 'libxcb-keysyms1', 'libxcb-render-util0', 'libxcb-xinerama0', 'libxcb-xkb1', 'libxkbcommon-x11-0']
    subprocess.run(['apt-get', '-qq', 'install', '-y'] + deps, check=True)

    print("⚙️ Running installer...")
    # Creates directory to avoid desktop entry error
    Path('/root/.local/share/applications').mkdir(parents=True, exist_ok=True)

    subprocess.run([str(installer), 'install', '-c', '--al', '-t', str(install_path)], check=True)
    installer.unlink() # Cleanup
    print(f"✅ Installation completed in {install_path}")

install_energyplus(EPLUS_URL, INSTALL_PATH)

# Adds to path immediately for this session
if str(INSTALL_PATH) not in sys.path:
    sys.path.insert(0, str(INSTALL_PATH))


In [ ]:
# @title 5. Configure Python API (no external module)
import os
from pathlib import Path

try:
    install_path = Path(INSTALL_PATH)
    if not install_path.exists():
        raise FileNotFoundError(f"Installation directory not found: {install_path}")

    os.environ.setdefault("EPLUS_HOME", str(install_path))
    if str(install_path) not in sys.path:
        sys.path.insert(0, str(install_path))
        print(f"✅ EnergyPlus API linked to Python: {install_path}")
    else:
        print("ℹ️  API already configured in the path.")

    from pyenergyplus.api import EnergyPlusAPI
    api = EnergyPlusAPI()
    print(f"✅ API Successfully Loaded! Engine Version: {api.functional.ep_version()}")

except FileNotFoundError as e:
    print(f"❌ {e}")
    print("   Ensure cell 4 (Install EnergyPlus) ran correctly.")
except Exception as e:
    print(f"❌ Fatal API error: {e}")


## ▶️ Simulation Execution
1. **Stage-In**: Download from GCS to `/tmp`.
2. **Run**: Execution via `pyenergyplus`.
3. **Stage-Out**: Upload results to GCS.

In [ ]:
# @title 6. Stage-In (Download Inputs)
from google.cloud import storage

WORK_DIR = Path('/tmp/energyplus_sim')
WORK_DIR.mkdir(exist_ok=True)

def download_inputs(bucket_name: str, files: dict[str, str], dest_dir: Path) -> dict[str, str]:
    """
    Download files from GCS to the local directory.

    Args:
        bucket_name: GCS bucket name
        files: Dictionary {type: bucket_path} e.g. {'idf': 'models/file.idf'}
        dest_dir: Local destination directory

    Returns:
        Dictionary with local paths of downloaded files
    """
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(bucket_name)

    local_files = {}
    print(f"⬇️ Downloading files from gs://{bucket_name}...")

    for file_type, gcs_path in files.items():
        blob = bucket.blob(gcs_path)
        # Saves only with filename (without folder structure)
        filename = Path(gcs_path).name
        dest = dest_dir / filename

        blob.download_to_filename(str(dest))
        local_files[file_type] = str(dest)
        print(f"  ✓ {gcs_path} → {filename}")

    return local_files

# Input files download
input_files = {
    'idf': IDF_FILE,
    'epw': EPW_FILE
}

local_paths = download_inputs(BUCKET_NAME, input_files, WORK_DIR)
local_idf = local_paths['idf']
local_epw = local_paths['epw']

print(f"\n📁 Local files ready:")
print(f"  IDF: {local_idf}")
print(f"  EPW: {local_epw}")


In [ ]:
# @title 7. Run Simulation (API)
api = EnergyPlusAPI()
state = api.state_manager.new_state()

args = [
    '-d', str(WORK_DIR),
    '-w', local_epw,
    local_idf
]

print(f"🚀 Starting simulation of {IDF_FILE}...")
exit_code = api.runtime.run_energyplus(state, args)

if exit_code == 0:
    print("\n✅ SIMULATION SUCCESSFUL!")
else:
    print(f"\n❌ Simulation failed. Code: {exit_code}")

api.state_manager.delete_state(state)


In [ ]:
# @title 8. Stage-Out (Upload Results)
from datetime import datetime

def upload_results(bucket_name: str, source_dir: Path, prefix: str) -> int:
    """
    Upload results to GCS bucket.

    Args:
        bucket_name: GCS bucket name
        source_dir: Local directory with results
        prefix: Bucket prefix (e.g. 'results/colab_vs_code_simulation_20260206_120000')

    Returns:
        Number of uploaded files
    """
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(bucket_name)

    print(f"⬆️ Uploading results to gs://{bucket_name}/{prefix}...")
    count = 0

    for file_path in source_dir.iterdir():
        if file_path.is_file() and not file_path.name.endswith(('.idf', '.epw')):
            # Ignores inputs, uploads only EnergyPlus outputs
            blob = bucket.blob(f"{prefix}/{file_path.name}")
            blob.upload_from_filename(str(file_path))
            print(f"  ✓ {file_path.name}")
            count += 1

    return count

if exit_code == 0:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    # New structure: results/colab_vs_code_simulation_{timestamp}/
    gcs_prefix = f"results/colab_vs_code_simulation_{timestamp}"

    files_uploaded = upload_results(BUCKET_NAME, WORK_DIR, gcs_prefix)

    print(f"\n✅ {files_uploaded} results files successfully uploaded!")
    print(f"\n🔗 Results available at:")
    print(f"   GCS Browser: https://console.cloud.google.com/storage/browser/{BUCKET_NAME}/{gcs_prefix}")
    print(f"   gsutil: gsutil ls -lh gs://{BUCKET_NAME}/{gcs_prefix}/")
else:
    print("⚠️ Simulation failed, upload cancelled.")

In [ ]:
# @title 9. View HTML Report
from IPython.display import HTML, display

report = WORK_DIR / 'eplustbl.htm'
if report.exists():
    # Reads only the beginning of the file to avoid crashing the browser with giant tables
    with open(report, 'r', encoding='utf-8', errors='replace') as f:
        content = f.read(5000) # First 5000 characters (Header + Summary)

    print("📄 Viewing report preview (truncated):")
    display(HTML(content + "...<br><br><b>⚠️ Full report available in the GCS Bucket (link above).</b>"))
else:
    print("⚠️ HTML Report not found.")


In [ ]:
# @title 10. Quick Visualization (Temperatures)
import pandas as pd
import matplotlib.pyplot as plt

csv_path = WORK_DIR / 'eplusout.csv'

# Generates the Chart
if csv_path.exists():
    print("📊 Generating temperature chart...")
    try:
        # Reads CSV ignoring common E+ formatting errors
        df = pd.read_csv(csv_path)

        # Searches for columns containing 'Temperature' and 'Zone' (Case insensitive)
        temp_cols = [c for c in df.columns if 'Temperature' in c and 'Zone' in c]

        if temp_cols:
            plt.figure(figsize=(15, 6))
            # Plots the first 5 zones found
            for col in temp_cols[:5]:
                plt.plot(df[col], label=col)

            plt.title("Zone Temperatures Profile")
            plt.xlabel("Time Step")
            plt.ylabel("Temperature (°C)")
            plt.legend(loc='best', fontsize='small')
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
        else:
            print("ℹ️ No zone temperature column found.")
    except Exception as e:
        print(f"❌ Error plotting: {e}")
else:
    print("⚠️ Results file not found.")


In [ ]:
# @title 11. Utilities to Explore the Bucket
import subprocess
from google.cloud import storage
from typing import List

def list_bucket_structure(bucket_name: str) -> None:
    """Lists the bucket's folder structure."""
    print(f"📁 Bucket structure gs://{bucket_name}:\n")

    folders = ['models/', 'weather/', 'results/', 'scripts/', 'notebooks/']

    for folder in folders:
        print(f"\n📂 {folder}")
        result = subprocess.run(
            ['gsutil', 'ls', f'gs://{bucket_name}/{folder}'],
            capture_output=True,
            text=True
        )
        if result.returncode == 0:
            files = [line.strip() for line in result.stdout.split('\n') if line.strip()]
            if files:
                for file in files[:10]:  # Shows up to 10 files
                    filename = file.split('/')[-1]
                    if filename:
                        print(f"  • {filename}")
                if len(files) > 10:
                    print(f"  ... and {len(files) - 10} more files")
            else:
                print("  (empty)")
        else:
            print(f"  ⚠️ Could not list")

def list_available_models(bucket_name: str) -> List[str]:
    """Lists available IDF models."""
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(bucket_name)

    blobs = bucket.list_blobs(prefix='models/')
    models = [blob.name for blob in blobs if blob.name.endswith('.idf')]

    print("📋 Available IDF models:")
    for model in models:
        print(f"  • {model}")

    return models

def list_available_weather(bucket_name: str) -> List[str]:
    """Lists available EPW files."""
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(bucket_name)

    blobs = bucket.list_blobs(prefix='weather/')
    weather_files = [blob.name for blob in blobs if blob.name.endswith('.epw')]

    print("🌦️  Available weather files:")
    for wf in weather_files:
        print(f"  • {wf}")

    return weather_files

def list_recent_results(bucket_name: str, limit: int = 5) -> None:
    """Lists the latest simulation results."""
    result = subprocess.run(
        ['gsutil', 'ls', f'gs://{bucket_name}/results/'],
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        folders = [line.strip() for line in result.stdout.split('\n') if 'simulation_' in line]
        folders.sort(reverse=True)  # Newest first

        print(f"📊 Latest {min(limit, len(folders))} simulations:")
        for folder in folders[:limit]:
            sim_name = folder.rstrip('/').split('/')[-1]
            print(f"  • {sim_name}")
            print(f"    {folder}")
    else:
        print("⚠️ Could not list results")

# Uncomment the functions you want to run:
print("💡 Available functions:")
print("  • list_bucket_structure(BUCKET_NAME)")
print("  • list_available_models(BUCKET_NAME)")
print("  • list_available_weather(BUCKET_NAME)")
print("  • list_recent_results(BUCKET_NAME)")
print("\nUsage example:")
print("  list_bucket_structure(BUCKET_NAME)")


## 📚 GCS Bucket Structure

The `eplus-colab-cloud-data` bucket is organized as follows:

```
gs://eplus-colab-cloud-data/
├── models/              # IDF Files (building models)
│   └── 5ZoneAirCooled.idf
├── weather/             # EPW Files (weather data)
│   └── USA_IL_Chicago-OHare.Intl.AP.725300_TMY3.epw
├── results/             # Simulation outputs (organized by timestamp)
│   ├── colab_vs_code_simulation_20260206_120000/
│   └── cloud_shell_simulation_20260206_195217/
├── scripts/             # Installation and automation scripts
│   └── install_energyplus.sh
└── notebooks/           # Archived notebooks
    └── _legacy/
```

### 🔄 Workflow

1. **Models and Weather**: Stored in `models/` and `weather/`
2. **Execution**: Notebook downloads files, runs simulation locally
3. **Results**: Uploaded automatically to `results/colab_vs_code_simulation_{timestamp}/`

### 📝 How to Add New Files

```bash
# Upload a new IDF model
gsutil cp my_model.idf gs://eplus-colab-cloud-data/models/

# Upload a weather file
gsutil cp city.epw gs://eplus-colab-cloud-data/weather/

# List results
gsutil ls -lh gs://eplus-colab-cloud-data/results/
```

In [ ]:
# @title 12. [EXAMPLE] Explore Available Files
# Run this cell to see all available files in the bucket

print("="*60)
print("🔍 EXPLORING THE GCS BUCKET")
print("="*60)

# Lists available models
print("\n")
models = list_available_models(BUCKET_NAME)

# Lists available weather files
print("\n")
weather = list_available_weather(BUCKET_NAME)

# Lists recent results
print("\n")
list_recent_results(BUCKET_NAME, limit=10)

print("\n" + "="*60)
print(f"✅ Total: {len(models)} models, {len(weather)} weather files")
print("="*60)


## 📚 Complete Guide: Adding New Files to the Bucket

### 🎯 Objective
Add new IDF models and weather files (EPW) to the GCS bucket for use in future simulations.

### 📋 Prerequisites
- **gsutil** installed and configured (already done in this notebook)
- **gcloud CLI** authenticated (done in Cell 1)
- Access to the `gs://eplus-colab-cloud-data` bucket

---

## 🔧 Method 1: Upload via gsutil (Recommended - Local)

### For IDF Models:
```bash
# Upload a single model
gsutil cp my_model.idf gs://eplus-colab-cloud-data/models/

# Upload multiple models from a folder
gsutil -m cp /local/path/models/*.idf gs://eplus-colab-cloud-data/models/

# Copy with overwrite (if it already exists)
gsutil -m cp -r /local/path/models/* gs://eplus-colab-cloud-data/models/
```

### For Weather Files (EPW):
```bash
# Upload a single weather file
gsutil cp chicago.epw gs://eplus-colab-cloud-data/weather/

# Upload multiple weather files
gsutil -m cp /local/path/weather/*.epw gs://eplus-colab-cloud-data/weather/

# Copy recursively (maintains folder structure)
gsutil -m cp -r /local/path/weather/* gs://eplus-colab-cloud-data/weather/
```

### For Results/Scripts:
```bash
# Upload scripts
gsutil cp my_script.py gs://eplus-colab-cloud-data/scripts/

# Upload notebooks
gsutil cp my_notebook.ipynb gs://eplus-colab-cloud-data/notebooks/
```

**Useful flags:**
- `-m`: Parallel upload (faster for multiple files)
- `-r`: Recursive (for folders)
- `-C`: Continue on error
- `-h`: Custom HTTP headers (e.g., metadata)

---

## 🐍 Method 2: Upload via Python (In Notebook)

You can run the code below directly in the notebook to upload:

```python
from google.cloud import storage
from pathlib import Path

PROJECT_ID = 'eplus-colab-cloud'
BUCKET_NAME = 'eplus-colab-cloud-data'

def upload_file_to_gcs(local_path: str, gcs_folder: str):
    """Uploads a file to the GCS bucket."""
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(BUCKET_NAME)
    
    local_file = Path(local_path)
    if not local_file.exists():
        print(f"❌ File not found: {local_path}")
        return
    
    # Creates the path in the bucket
    gcs_path = f"{gcs_folder.rstrip('/')}/{local_file.name}"
    blob = bucket.blob(gcs_path)
    
    # Upload
    print(f"⬆️ Uploading {local_file.name}...")
    blob.upload_from_filename(str(local_file))
    print(f"✅ Uploaded to gs://{BUCKET_NAME}/{gcs_path}")

# Usage examples:
# upload_file_to_gcs("my_model.idf", "models")
# upload_file_to_gcs("chicago.epw", "weather")
```

---

## 📂 Expected Bucket Structure

Keep the structure organized:

```
gs://eplus-colab-cloud-data/
├── models/
│   ├── 5ZoneAirCooled.idf          ← IDF Models
│   ├── Office_Medium.idf
│   └── Hospital_Large.idf
├── weather/
│   ├── USA_IL_Chicago-OHare...epw  ← Weather files
│   ├── USA_CA_Los_Angeles...epw
│   └── BRA_RJ_Rio_de_Janeiro...epw
├── results/
│   ├── simulation_vscode_20260206_211001/
│   └── simulation_vscode_20260206_180000/
├── scripts/
│   └── install_energyplus.sh
└── notebooks/
    └── EnergyPlus_VS_Code_Colab.ipynb
```

---

## 🔍 Verify Uploads

### Via gsutil:
```bash
# List files in models/
gsutil ls -lh gs://eplus-colab-cloud-data/models/

# List files in weather/
gsutil ls -lh gs://eplus-colab-cloud-data/weather/

# Check total bucket size
gsutil du -sh gs://eplus-colab-cloud-data/
```

### Via Python (in notebook):
```python
def list_files_in_folder(bucket_name: str, folder: str):
    client = storage.Client(project=PROJECT_ID)
    blobs = client.list_blobs(bucket_name, prefix=folder)
    
    print(f"📂 Files in {folder}:")
    for blob in blobs:
        size_mb = blob.size / (1024 * 1024)
        print(f"  • {blob.name} ({size_mb:.2f} MB)")

# Examples:
# list_files_in_folder(BUCKET_NAME, "models")
# list_files_in_folder(BUCKET_NAME, "weather")
```

---

## 🔄 Workflow for New Simulations

### Step 1: Prepare Files Locally
1. Obtain IDF model (from `energyplus-weather.run` or create new)
2. Obtain EPW weather file (from `energyplus-weather.run`)
3. Validate file structure

### Step 2: Upload to Bucket
```bash
gsutil cp my_new_model.idf gs://eplus-colab-cloud-data/models/
gsutil cp new_city.epw gs://eplus-colab-cloud-data/weather/
```

### Step 3: Update Cell 3 in Notebook
```python
IDF_FILE = 'models/my_new_model.idf'
EPW_FILE = 'weather/new_city.epw'
```

### Step 4: Execute Pipeline (Cells 6-8)
- Cell 6: Download new files
- Cell 7: Run simulation
- Cell 8: Upload results

---

## 💡 Tips & Best Practices

### ✅ What to Do:
- **Organize by folder**: Keep `models/`, `weather/`, `results/` separate
- **Descriptive names**: Use clear names (e.g., `Chicago_TMY3.epw` instead of `weather1.epw`)
- **Versioning**: If updating a file, add a timestamp (e.g., `model_v2_20260206.idf`)
- **Documentation**: Create a README.md in the bucket with file metadata
- **Compression**: For large uploads, compress with `.tar.gz` first

### ❌ What to Avoid:
- Do not delete input files while simulations are running
- Do not use names with special characters or spaces
- Do not upload intermediate files (binaries, caches)
- Do not mix inputs with outputs in the same folder

---

## 🎓 Complete Example: Adding a New Model

**Scenario**: You have a new model called `Hotel_5Star.idf` and weather data from `Miami.epw`

### Via Local Terminal:
```bash
# Login to GCP (if not already)
gcloud auth login

# Configure project
gcloud config set project eplus-colab-cloud

# Upload files
gsutil cp Hotel_5Star.idf gs://eplus-colab-cloud-data/models/
gsutil cp Miami_TMY3.epw gs://eplus-colab-cloud-data/weather/

# Verify upload
gsutil ls -lh gs://eplus-colab-cloud-data/models/Hotel_5Star.idf
gsutil ls -lh gs://eplus-colab-cloud-data/weather/Miami_TMY3.epw
```

### Via Notebook (Cell 6-8):
```python
# Edit Cell 3
IDF_FILE = 'models/Hotel_5Star.idf'
EPW_FILE = 'weather/Miami_TMY3.epw'

# Run Cells 6, 7, 8 normally
# Results will go to:
# gs://eplus-colab-cloud-data/results/simulation_vscode_TIMESTAMP/
```

---

## 🔗 Useful Resources

- **IDF Models**: [EnergyPlus Example Files](https://energyplus.net/weather/download)
- **Weather Data**: [IWEC Weather Data](https://energyplus.net/weather)
- **EnergyPlus Documentation**: [energyplus.net](https://energyplus.net/)
- **Google Cloud Storage**: [cloud.google.com/storage/docs](https://cloud.google.com/storage/docs)

---

## 📞 Troubleshooting

| Problem | Solution |
|----------|---------|
| `gsutil: command not found` | Install Google Cloud SDK: `curl https://sdk.cloud.google.com \| bash` |
| `AccessDenied` when uploading | Verify permissions: `gsutil acl ch -u $(gcloud config get-value account):O gs://eplus-colab-cloud-data` |
| File not found after upload | Wait 1-2 minutes (list cache) or use `gsutil ls` with `--stat` |
| Very slow upload | Use `-m` flag to parallelize or compress file first |

